In [ ]:
%%capture
!pip install unsloth
!pip install --upgrade pillow

In [1]:
# Authenticate with Hugging Face
from huggingface_hub import login

# Method 1: Kaggle Secrets (recommended)
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    hf_token = secrets.get_secret("HF_TOKEN")
    login(token=hf_token)
    print("Logged in via Kaggle Secrets")
except Exception:
    print("Could not find HF_TOKEN in Kaggle Secrets. Set it up or login manually.")

Logged in via Kaggle Secrets


In [2]:
from huggingface_hub import snapshot_download

path = snapshot_download(
    repo_id="unsloth/gemma-4-e4b-it",
    local_dir="/kaggle/working/gemma4-unsloth",
    ignore_patterns=["*.md"],
)
print(f"Downloaded to: {path}")


Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Downloaded to: /kaggle/working/gemma4-unsloth


In [ ]:
from unsloth import FastVisionModel
import torch, os

model, tokenizer = FastVisionModel.from_pretrained(
    "/kaggle/working/gemma4-unsloth",
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)

print("✅ Model loaded")
print(f"GPU memory used: {torch.cuda.memory_allocated()/1e9:.1f} GB")

In [ ]:
from PIL import Image, ImageDraw

def make_test_image(text, color):
    img = Image.new("RGB", (400, 300), color=color)
    draw = ImageDraw.Draw(img)
    draw.text((50, 130), text, fill="black")
    return img

images = [
    make_test_image("KPI: 14.6M visitors", (255, 240, 240)),
    make_test_image("Bar chart: left 30%, right 70%", (240, 255, 240)),
    make_test_image("Trend: up from 2022 to 2024", (240, 240, 255)),
]

def to_conversation(img, answer):
    return {
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": img},
                    {"type": "text", "text": "Describe this chart."},
                ],
            },
            {
                "role": "assistant",
                "content": [{"type": "text", "text": answer}],
            },
        ]
    }

dataset = [
    to_conversation(images[0], "This dashboard shows 14.6M visitors."),
    to_conversation(images[1], "Bar chart shows left at 30%, right at 70%."),
    to_conversation(images[2], "Trend is upward from 2022 to 2024."),
]

print(f"✅ Dataset created: {len(dataset)} examples")

In [8]:
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=dataset,
    args=SFTConfig(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        max_steps=3,
        learning_rate=2e-4,
        output_dir="./test_output",
        max_seq_length=2048,
        dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
    ),
)

print(f"GPU before training: {torch.cuda.memory_allocated()/1e9:.1f} GB")
trainer.train()
print("✅ Training ran without crash")
print(f"GPU after training: {torch.cuda.memory_allocated()/1e9:.1f} GB")



Unsloth: Model does not have a default image size - using 512
GPU before training: 10.2 GB


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3 | Num Epochs = 3 | Total steps = 3
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 36,700,160 of 8,032,856,608 (0.46% trained)
Caching is incompatible with gradient checkpointing in Gemma4TextDecoderLayer. Setting `past_key_values=None`.


Step,Training Loss
1,15.282346
2,15.282346
3,15.282346


✅ Training ran without crash
GPU after training: 10.2 GB
